In [68]:
# 필요한 패키지
# pip install sqlalchemy pymysql pandas

import pandas as pd
from sqlalchemy import create_engine, text
from typing import Sequence, Optional

# ── DB 접속 정보 ───────────────────────────────────────────────────────────────
from DATA.stock_invest_function import get_db_host

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

TABLE_NAME = "Korea_company_valuation_ver2"  # 스키마: investar.TABLE_NAME

def make_engine(db):
    url = (
        f"mysql+pymysql://{db['user']}:{db['password']}"
        f"@{db['host']}:{db['port']}/{db['database']}?charset=utf8mb4"
    )
    return create_engine(url, pool_pre_ping=True, future=True)

engine = make_engine(db_info)

# 1) forecast_date의 unique 값 추출
def get_unique_forecast_dates(include_null=False):
    q = f"""
        SELECT DISTINCT forecast_date
        FROM {TABLE_NAME}
        {"WHERE forecast_date IS NOT NULL" if not include_null else ""}
        ORDER BY forecast_date
    """
    with engine.begin() as conn:
        df = pd.read_sql(q, conn, parse_dates=["forecast_date"])
    return df["forecast_date"]

# 2) (ticker, forecast_date, keyword)로 indicator에 keyword가 포함된 값 조회 + date 기준 정렬
#    여러 indicator가 매칭되면 행으로 반환(롱 포맷). wide=True면 indicator별 칼럼으로 피벗.
def get_series_by_keyword(ticker, forecast_date, keyword, wide=False):
    sql = text(f"""
        SELECT `date`, `ticker`, `indicator`, `value`, `forecast_date`
        FROM {TABLE_NAME}
        WHERE ticker = :ticker
          AND forecast_date = :fdate
          AND indicator LIKE :kw
        ORDER BY `date`
    """)
    with engine.begin() as conn:
        df = pd.read_sql(
            sql, conn,
            params={"ticker": ticker, "fdate": forecast_date, "kw": f"%{keyword}%"},
            parse_dates=["date", "forecast_date"]
        )
    # 숫자형 보정
    if not df.empty:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
    if wide and not df.empty:
        df_wide = df.pivot_table(index="date", columns="indicator", values="value", aggfunc="last").sort_index()
        df_wide = df_wide.rename_axis(None, axis=1)
        return df_wide
    return df  # 롱 포맷: date, indicator, value …

# 3) indicator의 unique 값 추출
def get_unique_indicators(keyword=None):
    cond = "" if not keyword else "WHERE indicator LIKE :kw"
    sql = text(f"SELECT DISTINCT indicator FROM {TABLE_NAME} {cond} ORDER BY indicator")
    with engine.begin() as conn:
        df = pd.read_sql(sql, conn, params=(None if not keyword else {"kw": f"%{keyword}%"}))
    return df["indicator"]

# 4) (ticker, indicator, forecast_date 두 개) 입력 시 두 기간 차이 비교
#    반환: date 기준 병합(outer), col: value_fd1, value_fd2, diff = fd2 - fd1
def compare_indicator_between_dates(ticker, indicator, forecast_date_1, forecast_date_2):
    base_sql = text(f"""
        SELECT `date`, `value`
        FROM {TABLE_NAME}
        WHERE ticker = :ticker
          AND indicator = :indicator
          AND forecast_date = :fdate
        ORDER BY `date`
    """)
    with engine.begin() as conn:
        df1 = pd.read_sql(
            base_sql, conn,
            params={"ticker": ticker, "indicator": indicator, "fdate": forecast_date_1},
            parse_dates=["date"]
        )
        df2 = pd.read_sql(
            base_sql, conn,
            params={"ticker": ticker, "indicator": indicator, "fdate": forecast_date_2},
            parse_dates=["date"]
        )

    # 숫자형 보정
    for d in (df1, df2):
        if not d.empty:
            d["value"] = pd.to_numeric(d["value"], errors="coerce")

    df1 = df1.rename(columns={"value": f"value_{pd.to_datetime(forecast_date_1).date()}"})
    df2 = df2.rename(columns={"value": f"value_{pd.to_datetime(forecast_date_2).date()}"})

    out = pd.merge(df1, df2, on="date", how="outer").sort_values("date").set_index("date")
    if out.shape[1] == 2:
        cols = out.columns.tolist()
        out["diff"] = out[cols[1]] - out[cols[0]]  # fd2 - fd1
    return out

# -*- coding: utf-8 -*-
from __future__ import annotations
import pandas as pd
from typing import Optional, Union, Sequence, Tuple
from sqlalchemy import create_engine, text

# ─────────────────────────────────────────────────────────────────────────────

def make_engine(db_info: dict):
    url = (
        "mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
        "?charset=utf8mb4"
    ).format(**db_info)
    return create_engine(url, pool_recycle=3600, pool_pre_ping=True)


# ─────────────────────────────────────────────────────────────────────────────
# 1) 특정 forecast_date로 유니크 티커 조회
#   - forecast_date가 None → forecast_date IS NULL 조건
#   - forecast_date가 'YYYY-MM-DD' 나 'YYYY-MM-DD HH:MM:SS' → 해당 일자/시각 매칭
#   - '같은 날'로 묶고 싶다면 use_date_only=True 로 DATE(forecast_date)=DATE(:dt)
# ─────────────────────────────────────────────────────────────────────────────
def get_unique_tickers_by_forecast_date(
    db_info: dict,
    forecast_date: Optional[str] = None,
    table_name: str = "valuation_forecast_result",
    use_date_only: bool = True,
) -> pd.Series:
    """
    Return: pd.Series of unique tickers (name='ticker')
    """
    engine = make_engine(db_info)
    with engine.connect() as conn:
        if forecast_date is None:
            sql = text(f"""
                SELECT DISTINCT ticker
                FROM {table_name}
                WHERE forecast_date IS NULL
                ORDER BY ticker
            """)
            df = pd.read_sql(sql, conn)
        else:
            if use_date_only:
                sql = text(f"""
                    SELECT DISTINCT ticker
                    FROM {table_name}
                    WHERE DATE(forecast_date) = DATE(:dt)
                    ORDER BY ticker
                """)
            else:
                sql = text(f"""
                    SELECT DISTINCT ticker
                    FROM {table_name}
                    WHERE forecast_date = :dt
                    ORDER BY ticker
                """)
            df = pd.read_sql(sql, conn, params={"dt": forecast_date})

    return df["ticker"]


# ─────────────────────────────────────────────────────────────────────────────
# 2) indicator별 date1→date2 변화율 계산
#   - indicator: str 또는 [str, ...]  (None/'all'은 전체)
#   - 변화율 = (v2 / v1 - 1).  v1=0 또는 결측은 안전하게 제외
#   - 결과 정렬: pct_change(%) 내림차순
# Columns:
#   ['indicator','ticker','date1','date2','value_date1','value_date2',
#    'abs_change','pct_change']
# ─────────────────────────────────────────────────────────────────────────────

def get_indicator_change_rates(
    db_info: dict,
    forecast_date: Optional[str],
    indicator: Optional[Union[str, Sequence[str]]] = None,
    date1: str = "2027-03-31",
    date2: str = "2027-06-30",
    table_name: str = "valuation_forecast_result",
    use_date_only_for_forecast: bool = True,
    drop_zero_base: bool = True,
    tolerance_days: int = 3,
) -> pd.DataFrame:
    """
    안정화 버전: date1/date2 근사 매칭 + KeyError 방지
    """
    engine = make_engine(db_info)
    with engine.connect() as conn:
        conds, params = [], {}

        # forecast_date filter (SQL)
        if forecast_date is None:
            conds.append("forecast_date IS NULL")
        else:
            if use_date_only_for_forecast:
                conds.append("DATE(forecast_date) = DATE(:fdt)")
            else:
                conds.append("forecast_date = :fdt")
            params["fdt"] = forecast_date

        # indicator filter
        if indicator is None or (isinstance(indicator, str) and indicator.lower() == "all"):
            pass
        else:
            if isinstance(indicator, str):
                indicator_list = [indicator]
            else:
                indicator_list = list(indicator)
            placeholders = []
            for i, it in enumerate(indicator_list):
                key = f"i{i}"
                params[key] = it
                placeholders.append(f":{key}")
            conds.append(f"indicator IN ({', '.join(placeholders)})")

        where_sql = " AND ".join(conds) if conds else "1=1"

        # ---- SQL (DATE 변환 제거) ----
        sql = text(f"""
            SELECT
                `date` AS d,
                `ticker`,
                `indicator`,
                CAST(REPLACE(`value`, ',', '') AS DECIMAL(38,8)) AS value
            FROM {table_name}
            WHERE {where_sql}
              AND `date` IS NOT NULL
        """)
        raw = pd.read_sql(sql, conn, params=params, parse_dates=["d"])

    if raw.empty:
        print("⚠️ DB에서 forecast_date 조건에 맞는 데이터가 없습니다.")
        return pd.DataFrame(columns=[
            "indicator","ticker","date1","date2","value_date1","value_date2","abs_change","pct_change"
        ])

    # --- 전처리 ---
    raw["value"] = pd.to_numeric(raw["value"], errors="coerce")
    raw = raw.dropna(subset=["value"])
    if raw.empty:
        print("⚠️ value가 모두 결측이거나 숫자형 변환 실패.")
        return pd.DataFrame(columns=[
            "indicator","ticker","date1","date2","value_date1","value_date2","abs_change","pct_change"
        ])

    # 날짜 정규화
    raw["d"] = pd.to_datetime(raw["d"]).dt.floor("D")
    d1 = pd.to_datetime(date1)
    d2 = pd.to_datetime(date2)

    def near_mask(series, target, tol):
        delta = (series - target).abs().dt.days
        return delta <= tol

    mask = near_mask(raw["d"], d1, tolerance_days) | near_mask(raw["d"], d2, tolerance_days)
    raw2 = raw.loc[mask].copy()

    if raw2.empty:
        print(f"⚠️ date1={date1}, date2={date2} ±{tolerance_days}일 내 데이터 없음.")
        return pd.DataFrame(columns=[
            "indicator","ticker","date1","date2","value_date1","value_date2","abs_change","pct_change"
        ])

    # ---- pivot ----
    piv = (
        raw2
        .pivot_table(index=["indicator","ticker"], columns="d", values="value", aggfunc="last")
        .reset_index()
    )

    if piv.empty:
        print("⚠️ pivot 결과가 비었습니다.")
        return pd.DataFrame(columns=[
            "indicator","ticker","date1","date2","value_date1","value_date2","abs_change","pct_change"
        ])

    print(f"✅ pivot 컬럼 목록:\n{list(piv.columns)}")  # 디버그용

    # ---- 날짜 컬럼 실제 존재 확인 ----
    date_cols = [c for c in piv.columns if isinstance(c, pd.Timestamp)]
    if not date_cols:
        print("⚠️ pivot에 날짜 컬럼이 없습니다.")
        return pd.DataFrame(columns=[
            "indicator","ticker","date1","date2","value_date1","value_date2","abs_change","pct_change"
        ])

    def get_closest_col(df_cols, target):
        deltas = [abs((c - target).days) for c in df_cols]
        return df_cols[deltas.index(min(deltas))]

    d1_actual = get_closest_col(date_cols, d1)
    d2_actual = get_closest_col(date_cols, d2)
    print(f"📅 실제 사용 날짜: d1_actual={d1_actual.date()}, d2_actual={d2_actual.date()}")

    # ---- rename 안전 처리 ----
    rename_map = {}
    if d1_actual in piv.columns:
        rename_map[d1_actual] = "value_date1"
    if d2_actual in piv.columns:
        rename_map[d2_actual] = "value_date2"

    piv = piv.rename(columns=rename_map)

    # ✅ rename 후에도 컬럼이 없으면 생성
    if "value_date1" not in piv.columns:
        piv["value_date1"] = pd.NA
    if "value_date2" not in piv.columns:
        piv["value_date2"] = pd.NA

    out = piv[["indicator","ticker","value_date1","value_date2"]].copy()

    # 결측/0 제거
    out = out.dropna(subset=["value_date1","value_date2"])
    if drop_zero_base:
        out = out[out["value_date1"] != 0]

    if out.empty:
        print("⚠️ 유효한 변화 계산 대상이 없습니다.")
        return pd.DataFrame(columns=[
            "indicator","ticker","date1","date2","value_date1","value_date2","abs_change","pct_change"
        ])

    # ---- 변화율 계산 ----
    out["abs_change"] = out["value_date2"] - out["value_date1"]
    out["pct_change"] = (out["value_date2"] / out["value_date1"] - 1.0) * 100.0
    out.insert(2, "date1", d1_actual.date())
    out.insert(3, "date2", d2_actual.date())

    out = out.sort_values(["indicator","pct_change"], ascending=[True, False]).reset_index(drop=True)
    print(f"✅ 최종 결과 행 수: {len(out)}")
    return out


CHUNKSIZE  = 200_000

def make_engine(db):
    url = (
        f"mysql+pymysql://{db['user']}:{db['password']}"
        f"@{db['host']}:{db['port']}/{db['database']}?charset=utf8mb4"
    )
    return create_engine(url, pool_pre_ping=True, future=True)

engine = make_engine(db_info)

# ── 0) 로우 카운트 참고용 (옵션) ─────────────────────────────────────────────
with engine.begin() as conn:
    rowcount = pd.read_sql(
        text(f"SELECT COUNT(*) AS c FROM {TABLE_NAME}"), conn
    )["c"].iloc[0]
print(f"[INFO] total rows in {TABLE_NAME}: {rowcount:,}")

# ── 공통 SELECT (날짜 컬럼을 함께 들고 오기) ─────────────────────────────────
# 필요한 경우 * 로 바꾸세요. 아래는 가장 많이 쓰는 컬럼 예시입니다.
BASE_SQL = f"""
SELECT
  `date`,
  `ticker`,
  `indicator`,
  `value`,
  `forecast_date`
FROM {TABLE_NAME}
"""

# ============ (A) 메모리가 충분한 경우: 단일 DataFrame 로드 ==================
def load_full_table_as_dataframe(chunksize=CHUNKSIZE) -> pd.DataFrame:
    """
    테이블 전체를 청크로 읽어 합친 DataFrame 반환.
    메모리가 충분할 때만 사용하세요.
    """
    all_parts = []
    total = 0
    with engine.begin() as conn:
        for i, chunk in enumerate(
            pd.read_sql(
                text(BASE_SQL), conn,
                chunksize=chunksize,
                parse_dates=["date", "forecast_date"],  # 날짜 컬럼 파싱
            ),
            start=1
        ):
            total += len(chunk)
            all_parts.append(chunk)
            print(f"[CHUNK {i}] rows={len(chunk):,}  |  accumulated={total:,}")

    if not all_parts:
        print("[WARN] No rows read.")
        return pd.DataFrame(columns=["date","ticker","indicator","value","forecast_date"])

    df_all = pd.concat(all_parts, axis=0, ignore_index=True)
    print(f"[DONE] Loaded DataFrame with rows={len(df_all):,}")
    return df_all


# 필요시 교체
def make_engine(db):
    url = f"mysql+pymysql://{db['user']}:{db['password']}@{db['host']}:{db['port']}/{db['database']}?charset=utf8mb4"
    return create_engine(url, pool_pre_ping=True, future=True)

def get_last_market_cap(
    db_info: dict,
    table_name: str,
    ticker_list: Sequence[str],
    indicator_exact: Optional[str] = None,     # 예: '시가총액' 또는 'market_cap'
    indicator_like: Optional[str] = None,      # 예: '%시가총액%' 또는 'market_cap%'
) -> pd.DataFrame:
    """
    입력 티커들에 대해 '마지막 날짜'의 시가총액만 추출 (롱 포맷)
    반환 칼럼: ['ticker','date','last_market_cap']
    """

    if not ticker_list:
        return pd.DataFrame(columns=["ticker","date","last_market_cap"])

    engine = make_engine(db_info)
    params = {"tickers": tuple(ticker_list)}

    # indicator 조건
    ind_cond = "1=1"
    if indicator_exact:
        ind_cond = "indicator = :ind_exact"
        params["ind_exact"] = indicator_exact
    elif indicator_like:
        ind_cond = "indicator LIKE :ind_like"
        params["ind_like"] = indicator_like
    else:
        # 기본값: 한국어/영문 둘 다 시도
        ind_cond = "(indicator = '시가총액' OR indicator LIKE 'market_cap%')"

    # 1) 우선 윈도 함수(ROW_NUMBER) 사용 (MySQL 8+/MariaDB 10.2+)
    sql_win = text(f"""
        WITH base AS (
            SELECT
                `date`,
                `ticker`,
                CAST(REPLACE(`value`, ',', '') AS DECIMAL(38,8)) AS value
            FROM {table_name}
            WHERE ticker IN :tickers
              AND {ind_cond}
              AND `date` IS NOT NULL
        ),
        ranked AS (
            SELECT *,
                   ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY `date` DESC) AS rn
            FROM base
        )
        SELECT
            ticker,
            DATE(`date`) AS `date`,
            value AS last_market_cap
        FROM ranked
        WHERE rn = 1
        ORDER BY ticker
    """)

    try:
        with engine.begin() as conn:
            df = pd.read_sql(sql_win, conn, params=params, parse_dates=["date"])
        if not df.empty:
            return df[["ticker","date","last_market_cap"]]
    except Exception:
        pass  # 구버전 DB면 아래 조인 방식으로 폴백

    # 2) 폴백: 윈도 함수 없이 MAX(date) 조인
    sql_join = text(f"""
        SELECT b.ticker,
               DATE(b.`date`) AS `date`,
               b.value AS last_market_cap
        FROM (
            SELECT ticker, MAX(`date`) AS max_date
            FROM {table_name}
            WHERE ticker IN :tickers
              AND {ind_cond}
              AND `date` IS NOT NULL
            GROUP BY ticker
        ) m
        JOIN (
            SELECT
                `date`, `ticker`,
                CAST(REPLACE(`value`, ',', '') AS DECIMAL(38,8)) AS value
            FROM {table_name}
            WHERE ticker IN :tickers
              AND {ind_cond}
              AND `date` IS NOT NULL
        ) b
          ON b.ticker = m.ticker AND b.`date` = m.max_date
        ORDER BY b.ticker
    """)

    with engine.begin() as conn:
        df2 = pd.read_sql(sql_join, conn, params=params, parse_dates=["date"])
    return df2[["ticker","date","last_market_cap"]]

# ── 사용 예시 ─────────────────────────────────────────
# db_info = { 'host': ..., 'port': 3307, 'user': 'stox7412', 'password': '...', 'database': 'investar' }
# ticker_list = ["A005930","A000660","A035420"]
# df_last = get_last_market_cap(db_info, "Korea_company_valuation_ver2", ticker_list,
#                               indicator_exact="시가총액")  # 또는 indicator_like="%market_cap%"
# print(df_last)

[INFO] total rows in Korea_company_valuation_ver2: 252,197


In [29]:
print(get_unique_forecast_dates().tail())

0   2025-10-26
1   2025-10-29
2   2025-10-30
3   2025-10-31
4   2025-11-09
Name: forecast_date, dtype: datetime64[ns]


In [27]:
TICKER = "A131290"
FORECAST_DATE = "2025-11-09"

ex_long = get_series_by_keyword(ticker=TICKER , forecast_date=FORECAST_DATE, keyword="rev", wide=False)

In [28]:
ex_long.tail(15)

,date,ticker,indicator,value,forecast_date
843,2026-06-30,A131290,revenue_prophet_ttm,3.917621e+08,2025-11-09
844,2026-06-30,A131290,revenue_lstm_ttm,3.289232e+08,2025-11-09
845,2026-06-30,A131290,revenue_theta_ttm,4.522003e+08,2025-11-09
846,2026-09-30,A131290,revenue_sarima,1.734649e+08,2025-11-09
847,2026-09-30,A131290,revenue_sarima_exog,1.974335e+08,2025-11-09
848,2026-09-30,A131290,revenue_ets,1.347814e+08,2025-11-09
849,2026-09-30,A131290,revenue_prophet,1.019839e+08,2025-11-09
850,2026-09-30,A131290,revenue_lstm,8.575551e+07,2025-11-09
851,2026-09-30,A131290,revenue_theta,1.255493e+08,2025-11-09
852,2026-09-30,A131290,revenue_sarima_ttm,5.829439e+08,2025-11-09


In [5]:
# ex_long → indicator를 컬럼으로 피벗
ex_pivot = (
    ex_long
    .pivot_table(
        index=["date", "ticker"],      # 행 인덱스
        columns="indicator",           # 열로 변환할 컬럼
        values="value",                # 값으로 쓸 컬럼
        aggfunc="last"                 # 중복 시 마지막 값 사용
    )
    .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
)

# 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
ex_pivot.columns.name = None

# 확인
print(ex_pivot.head())


        date   ticker  revenue_ets  revenue_ets_ttm  revenue_lstm  \
0 2004-12-31  A131290          0.0              NaN           0.0   
1 2005-12-31  A131290          0.0              NaN           0.0   
2 2006-12-31  A131290          0.0              NaN           0.0   
3 2007-12-31  A131290          0.0              0.0           0.0   
4 2008-12-31  A131290          0.0              0.0           0.0   

   revenue_lstm_ttm  revenue_prophet  revenue_prophet_ttm  revenue_sarima  \
0               NaN              0.0                  NaN             0.0   
1               NaN              0.0                  NaN             0.0   
2               NaN              0.0                  NaN             0.0   
3               0.0              0.0                  0.0             0.0   
4               0.0              0.0                  0.0             0.0   

   revenue_sarima_exog  revenue_sarima_exog_ttm  revenue_sarima_ttm  \
0                  0.0                      NaN    

In [6]:
ex_pivot.columns.tolist()

['date',
 'ticker',
 'revenue_ets',
 'revenue_ets_ttm',
 'revenue_lstm',
 'revenue_lstm_ttm',
 'revenue_prophet',
 'revenue_prophet_ttm',
 'revenue_sarima',
 'revenue_sarima_exog',
 'revenue_sarima_exog_ttm',
 'revenue_sarima_ttm',
 'revenue_theta',
 'revenue_theta_ttm']

In [7]:
ex_pivot[['date', 'revenue_ets', 'revenue_prophet', 'revenue_sarima', 'revenue_sarima_exog', 'revenue_theta']].tail(10)

,date,revenue_ets,revenue_prophet,revenue_sarima,revenue_sarima_exog,revenue_theta
63,2024-06-30,7.604882e+07,7.604882e+07,7.604882e+07,7.604882e+07,7.604882e+07
64,2024-09-30,1.106759e+08,1.106759e+08,1.106759e+08,1.106759e+08,1.106759e+08
65,2024-12-31,1.031176e+08,1.031176e+08,1.031176e+08,1.031176e+08,1.031176e+08
66,2025-03-31,8.303146e+07,8.303146e+07,8.303146e+07,8.303146e+07,8.303146e+07
67,2025-06-30,1.176139e+08,1.176139e+08,1.176139e+08,1.176139e+08,1.176139e+08
68,2025-09-30,1.258462e+08,9.551803e+07,1.402246e+08,1.555271e+08,1.220416e+08
69,2025-12-31,1.164918e+08,9.714778e+07,1.393132e+08,1.355502e+08,1.159213e+08
70,2026-03-31,1.010367e+08,9.874211e+07,1.243319e+08,6.161133e+07,1.003556e+08
71,2026-06-30,1.193189e+08,1.003541e+08,1.458339e+08,1.227843e+08,1.138818e+08
72,2026-09-30,1.347814e+08,1.019839e+08,1.734649e+08,1.974335e+08,1.255493e+08


In [8]:
psr_long = get_series_by_keyword(ticker=TICKER, forecast_date= FORECAST_DATE, keyword="psr", wide=False)

# ex_long → indicator를 컬럼으로 피벗
psr_pivot = (
    psr_long
    .pivot_table(
        index=["date", "ticker"],      # 행 인덱스
        columns="indicator",           # 열로 변환할 컬럼
        values="value",                # 값으로 쓸 컬럼
        aggfunc="last"                 # 중복 시 마지막 값 사용
    )
    .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
)

# 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
psr_pivot.columns.name = None

# 확인
print(psr_pivot.head())

        date   ticker       psr  psr_ETS  psr_LSTM  psr_Prophet  \
0 2015-05-31  A131290  1.670988      NaN       NaN          NaN   
1 2015-06-30  A131290  1.425398      NaN       NaN          NaN   
2 2015-07-31  A131290  1.183269      NaN       NaN          NaN   
3 2015-08-31  A131290  1.413862      NaN       NaN          NaN   
4 2015-09-30  A131290  1.348606      NaN       NaN          NaN   

   psr_SARIMA_exog  psr_SARIMA_noexog  psr_Theta  
0              NaN                NaN        NaN  
1              NaN                NaN        NaN  
2              NaN                NaN        NaN  
3              NaN                NaN        NaN  
4              NaN                NaN        NaN  


In [9]:
psr_pivot.tail(20)

,date,ticker,psr,psr_ETS,psr_LSTM,psr_Prophet,psr_SARIMA_exog,psr_SARIMA_noexog,psr_Theta
120,2025-05-31,A131290,1.483180,NaN,NaN,NaN,NaN,NaN,NaN
121,2025-06-30,A131290,1.471999,NaN,NaN,NaN,NaN,NaN,NaN
122,2025-07-31,A131290,1.453368,NaN,NaN,NaN,NaN,NaN,NaN
123,2025-08-31,A131290,1.367372,NaN,NaN,NaN,NaN,NaN,NaN
124,2025-09-30,A131290,1.940901,NaN,NaN,NaN,NaN,NaN,NaN
125,2025-10-31,A131290,1.889921,NaN,NaN,NaN,NaN,NaN,NaN
126,2025-11-30,A131290,1.919052,NaN,NaN,NaN,NaN,NaN,NaN
127,2025-12-31,A131290,NaN,2.097501,1.690776,2.842225,2.126575,1.907624,2.142961
128,2026-01-31,A131290,NaN,2.083264,1.755577,2.809858,2.110641,1.907624,2.175255
129,2026-02-28,A131290,NaN,1.896754,1.828847,2.550771,1.933417,1.907624,1.951413


In [10]:
mc_long = get_series_by_keyword(ticker=TICKER, forecast_date=FORECAST_DATE, keyword="mc_", wide=False)

# ex_long → indicator를 컬럼으로 피벗
mc_pivot = (
    mc_long
    .pivot_table(
        index=["date", "ticker"],      # 행 인덱스
        columns="indicator",           # 열로 변환할 컬럼
        values="value",                # 값으로 쓸 컬럼
        aggfunc="last"                 # 중복 시 마지막 값 사용
    )
    .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
)

# 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
mc_pivot.columns.name = None

# 확인
print(mc_pivot.head())

        date   ticker        mc_ets       mc_lstm    mc_prophet  \
0 2025-12-31  A131290  9.291582e+08  6.131787e+08  1.117879e+09   
1 2026-01-31  A131290  9.228514e+08  6.366798e+08  1.105149e+09   
2 2026-02-28  A131290  8.402308e+08  6.632519e+08  1.003247e+09   
3 2026-03-31  A131290  9.019043e+08  6.822331e+08  1.094144e+09   
4 2026-04-30  A131290  9.945831e+08  7.012167e+08  1.220894e+09   

   mc_sarima_exog  mc_sarima_noexog      mc_theta  
0    1.045686e+09      9.160091e+08  9.399204e+08  
1    1.037850e+09      9.160091e+08  9.540851e+08  
2    9.507051e+08      9.160091e+08  8.559060e+08  
3    9.329692e+08      9.947948e+08  9.098817e+08  
4    1.027540e+09      9.947948e+08  9.899661e+08  


In [11]:
mc_pivot.tail(14)

,date,ticker,mc_ets,mc_lstm,mc_prophet,mc_sarima_exog,mc_sarima_noexog,mc_theta
0,2025-12-31,A131290,9.291582e+08,6.131787e+08,1.117879e+09,1.045686e+09,9.160091e+08,9.399204e+08
1,2026-01-31,A131290,9.228514e+08,6.366798e+08,1.105149e+09,1.037850e+09,9.160091e+08,9.540851e+08
2,2026-02-28,A131290,8.402308e+08,6.632519e+08,1.003247e+09,9.507051e+08,9.160091e+08,8.559060e+08
3,2026-03-31,A131290,9.019043e+08,6.822331e+08,1.094144e+09,9.329692e+08,9.947948e+08,9.098817e+08
4,2026-04-30,A131290,9.945831e+08,7.012167e+08,1.220894e+09,1.027540e+09,9.947948e+08,9.899661e+08
5,2026-05-31,A131290,1.002419e+09,7.091292e+08,1.225808e+09,1.033455e+09,9.947948e+08,1.017051e+09
6,2026-06-30,A131290,9.822109e+08,6.462659e+08,1.149710e+09,1.033270e+09,1.048628e+09,1.015218e+09
7,2026-07-31,A131290,1.028406e+09,6.484285e+08,1.203202e+09,1.078775e+09,1.048628e+09,1.070854e+09
8,2026-08-31,A131290,9.724809e+08,6.523850e+08,1.110149e+09,1.024364e+09,1.048628e+09,1.007905e+09
9,2026-09-30,A131290,9.707402e+08,6.661768e+08,1.111373e+09,1.090316e+09,1.112038e+09,9.597954e+08


In [30]:
# ── 2️⃣ 함수 실행 ───────────────────────────────────────────
df_change = get_indicator_change_rates(
    db_info=db_info,
    forecast_date="2025-11-09",            # 📌 이 forecast_date와 일치하는 데이터 전체를 SQL에서 추출
    indicator="mc_sarima_exog",            # 또는 None → 전체 indicator
    date1="2025-11-30",                    # 📈 비교 시작일
    date2="2026-11-30",                    # 📉 비교 종료일
    table_name="Korea_company_valuation_ver2",  # 실제 테이블명
    use_date_only_for_forecast=True,       # 날짜만 비교 (시각 무시)
    drop_zero_base=True                    # 0으로 나누는 행 제거
)

print(df_change.head())

Empty DataFrame
Columns: [indicator, ticker, date1, date2, value_date1, value_date2, abs_change, pct_change]
Index: []


In [24]:
tickers

0      A000270
1      A000500
2      A000660
3      A001440
4      A002350
5      A004000
6      A005380
7      A005930
8      A006400
9      A006910
10     A007700
11     A009150
12     A010120
13     A010140
14     A011780
15     A031980
16     A033500
17     A035420
18     A035720
19     A036190
20     A042370
21     A042660
22    A042700 
23     A043150
24     A044820
25     A051910
26     A059090
27     A060980
28     A068270
29     A071280
30     A071970
31     A073240
32     A077360
33     A082740
34     A084370
35     A086390
36     A093520
37     A095610
38     A103140
39     A103590
40     A105630
41     A114810
42     A123330
43     A123700
44     A131290
45     A140860
46     A161390
47     A207940
48     A214150
49     A232140
50     A253590
51     A375500
Name: ticker, dtype: object

In [66]:
# 2) 변화율 표 (예: forecast_date가 NULL, 모든 indicator, 2027-03-31 → 2027-06-30)
df_change = get_indicator_change_rates(
    db_info=db_info,
    forecast_date="2025-11-09",
    indicator="mc_sarima_exog",
    date1="2025-12-31",
    date2="2026-11-30",
    table_name="Korea_company_valuation_ver2",
    tolerance_days=10,     # 📌 10일 이내 근사 허용
)
print(df_change.head())


✅ pivot 컬럼 목록:
['indicator', 'ticker', Timestamp('2025-12-31 00:00:00'), Timestamp('2026-11-30 00:00:00')]
📅 실제 사용 날짜: d1_actual=2025-12-31, d2_actual=2026-11-30
✅ 최종 결과 행 수: 30
d       indicator   ticker       date1       date2   value_date1  \
0  mc_sarima_exog  A298040  2025-12-31  2026-11-30  2.596712e+10   
1  mc_sarima_exog  A006910  2025-12-31  2026-11-30  4.693914e+08   
2  mc_sarima_exog  A042660  2025-12-31  2026-11-30  5.570474e+10   
3  mc_sarima_exog  A000660  2025-12-31  2026-11-30  6.310504e+11   
4  mc_sarima_exog  A010120  2025-12-31  2026-11-30  1.649254e+10   

d   value_date2    abs_change  pct_change  
0  4.153605e+10  1.556893e+10   59.956311  
1  7.133279e+08  2.439364e+08   51.968658  
2  7.678052e+10  2.107578e+10   37.834802  
3  8.457157e+11  2.146654e+11   34.017154  
4  2.003094e+10  3.538401e+09   21.454558  


In [67]:
df_change

d,indicator,ticker,date1,date2,value_date1,value_date2,abs_change,pct_change
0,mc_sarima_exog,A298040,2025-12-31,2026-11-30,2.596712e+10,4.153605e+10,1.556893e+10,59.956311
1,mc_sarima_exog,A006910,2025-12-31,2026-11-30,4.693914e+08,7.133279e+08,2.439364e+08,51.968658
2,mc_sarima_exog,A042660,2025-12-31,2026-11-30,5.570474e+10,7.678052e+10,2.107578e+10,37.834802
3,mc_sarima_exog,A000660,2025-12-31,2026-11-30,6.310504e+11,8.457157e+11,2.146654e+11,34.017154
4,mc_sarima_exog,A010120,2025-12-31,2026-11-30,1.649254e+10,2.003094e+10,3.538401e+09,21.454558
5,mc_sarima_exog,A033500,2025-12-31,2026-11-30,1.342710e+09,1.610712e+09,2.680016e+08,19.959746
6,mc_sarima_exog,A044820,2025-12-31,2026-11-30,1.972659e+08,2.359670e+08,3.870113e+07,19.618762
7,mc_sarima_exog,A103140,2025-12-31,2026-11-30,3.669279e+09,4.320668e+09,6.513886e+08,17.752495
8,mc_sarima_exog,A114810,2025-12-31,2026-11-30,5.934595e+08,6.975365e+08,1.040770e+08,17.537342
9,mc_sarima_exog,A042700,2025-12-31,2026-11-30,1.429872e+10,1.672447e+10,2.425747e+09,16.964787


In [78]:

ticker_list = df_change["ticker"].unique().tolist()
df_mck = get_last_market_cap(db_info, "ks_listed_company_daily_marketcap", ticker_list,
                              indicator_exact="시가총액")  # 또는 indicator_like="%market_cap%"
df_mck['market_cap_adj'] = df_mck['last_market_cap']/1000
merged_df = pd.merge(df_change, df_mck[["ticker", "last_market_cap", "market_cap_adj"]], on=["ticker"], how="left")
merged_df['upside_potential'] = (merged_df['value_date2'] -  merged_df['market_cap_adj'])/merged_df['market_cap_adj']

In [79]:
merged_df

,indicator,ticker,date1,date2,value_date1,value_date2,abs_change,pct_change,last_market_cap,market_cap_adj,upside_potential
0,mc_sarima_exog,A298040,2025-12-31,2026-11-30,2.596712e+10,4.153605e+10,1.556893e+10,59.956311,2.040210e+13,2.040210e+10,1.035871
1,mc_sarima_exog,A006910,2025-12-31,2026-11-30,4.693914e+08,7.133279e+08,2.439364e+08,51.968658,2.505620e+11,2.505620e+08,1.846912
2,mc_sarima_exog,A042660,2025-12-31,2026-11-30,5.570474e+10,7.678052e+10,2.107578e+10,37.834802,3.885320e+13,3.885320e+10,0.976170
3,mc_sarima_exog,A000660,2025-12-31,2026-11-30,6.310504e+11,8.457157e+11,2.146654e+11,34.017154,4.222410e+14,4.222410e+11,1.002922
4,mc_sarima_exog,A010120,2025-12-31,2026-11-30,1.649254e+10,2.003094e+10,3.538401e+09,21.454558,1.320000e+13,1.320000e+10,0.517496
5,mc_sarima_exog,A033500,2025-12-31,2026-11-30,1.342710e+09,1.610712e+09,2.680016e+08,19.959746,9.521660e+11,9.521660e+08,0.691629
6,mc_sarima_exog,A044820,2025-12-31,2026-11-30,1.972659e+08,2.359670e+08,3.870113e+07,19.618762,1.374320e+11,1.374320e+08,0.716973
7,mc_sarima_exog,A103140,2025-12-31,2026-11-30,3.669279e+09,4.320668e+09,6.513886e+08,17.752495,2.774400e+12,2.774400e+09,0.557334
8,mc_sarima_exog,A114810,2025-12-31,2026-11-30,5.934595e+08,6.975365e+08,1.040770e+08,17.537342,4.320430e+11,4.320430e+08,0.614507
9,mc_sarima_exog,A042700,2025-12-31,2026-11-30,1.429872e+10,1.672447e+10,2.425747e+09,16.964787,1.212370e+13,1.212370e+10,0.379485


In [47]:
# TABLE_NAME = "Korea_company_valuation_ver2"
#   # 청크 크기: 환경에 맞게 조절 (예: 50k ~ 500k)
# df_all = load_full_table_as_dataframe()
# # forecast_date가 NaT가 아닌 행만 선택
# df_all = df_all[df_all["forecast_date"].notna()].copy()
#
# selected_data = df_all[df_all['forecast_date'] == '2025-11-09']
#
#
# # indicator를 컬럼으로 pivot
# df_pivot = (
#     selected_data
#     .pivot_table(
#         index=["date", "ticker"],   # 행 기준 (group by)
#         columns="indicator",        # 열로 표시할 항목
#         values="value",             # 값으로 쓸 컬럼
#         aggfunc="last"              # 중복시 마지막 값 사용 (mean, first 등도 가능)
#     )
#     .reset_index()                  # 계층 인덱스 해제
# )
#
# # 컬럼명 정리 (MultiIndex 방지)
# df_pivot.columns.name = None
#
#  # 1️⃣ date를 인덱스로 지정
# df_pivot = df_pivot.set_index("date")
#
# # "mc_" 가 포함된 컬럼만 필터링
# mc_cols = [col for col in df_pivot.columns if "mc_" in col]
#
# # 해당 컬럼만 추출 (date 인덱스, ticker 포함)
# df_mc = df_pivot[["ticker"] + mc_cols].copy()
#
# # 2️⃣ ticker 기준으로 정렬 (오름차순)
# df_mc = df_mc.sort_values(by=["ticker", "date"], ascending=[True, True])
#
# # 3️⃣ (선택) 인덱스를 다시 정렬된 상태로 보존하려면
# df_mc = df_mc.sort_index(level=0)
# # 모든 값이 NaN인 행 제거
# df_mc = df_mc.dropna(axis=0, how='all')

[CHUNK 1] rows=200,000  |  accumulated=200,000
[CHUNK 2] rows=52,197  |  accumulated=252,197
[DONE] Loaded DataFrame with rows=252,197
